In [1]:
# Import Library
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')
# Load Dataset
df = pd.read_csv('spotify_mood.csv')

df.head()
print("Jumlah Data :", df.shape)

df.info()
def mood_label(valence):
    if valence < 0.33:
        return "Sad"
    elif valence < 0.66:
        return "Calm"
    else:
        return "Happy"

df["mood"] = df["valence"].apply(mood_label)

df["mood"].value_counts()
features = [
    'danceability',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

X = df[features]
y = df['mood']
le = LabelEncoder()

y_encoded = le.fit_transform(y)

print(le.classes_)
['Calm', 'Happy', 'Sad']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("Accuracy :", round(acc*100,2), "%")
print(classification_report(y_test, y_pred))
import joblib

joblib.dump(rf, "spotify_mood_model.pkl")
joblib.dump(le, "label_encoder.pkl")

print("Model berhasil disimpan")
sample = [[
    0.80,  # danceability
    0.90,  # energy
    0.05,  # speechiness
    0.10,  # acousticness
    0.00,  # instrumentalness
    0.15,  # liveness
    0.85,  # valence
    120    # tempo
]]

pred = rf.predict(sample)

print("Mood :", le.inverse_transform(pred)[0])
import gradio as gr
import joblib

model = joblib.load("spotify_mood_model.pkl")
encoder = joblib.load("label_encoder.pkl")

def predict_mood(
    danceability,
    energy,
    speechiness,
    acousticness,
    instrumentalness,
    liveness,
    valence,
    tempo
):

    data = [[
        danceability,
        energy,
        speechiness,
        acousticness,
        instrumentalness,
        liveness,
        valence,
        tempo
    ]]

    pred = model.predict(data)

    mood = encoder.inverse_transform(pred)[0]

    return mood


interface = gr.Interface(
    fn=predict_mood,
    inputs=[
        gr.Slider(0,1,label="Danceability"),
        gr.Slider(0,1,label="Energy"),
        gr.Slider(0,1,label="Speechiness"),
        gr.Slider(0,1,label="Acousticness"),
        gr.Slider(0,1,label="Instrumentalness"),
        gr.Slider(0,1,label="Liveness"),
        gr.Slider(0,1,label="Valence"),
        gr.Number(label="Tempo")
    ],
    outputs="text",
    title="Spotify Mood Prediction",
    description="Prediksi Mood Lagu Menggunakan Random Forest"
)

interface.launch()


Jumlah Data : (114000, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float6